# 第五阶段：智能指针

## 实验 7：循环引用与 ownership graph

引用计数只能判断 strong owner 的数量，不能判断一组对象是否仍能被程序访问。当对象通过 `shared_ptr` 互相拥有时，外部 owner 即使全部消失，内部计数仍不会归零，这就是 cyclic ownership leak。

本实验关注：

- A↔B strong cycle 为什么无法触发析构；
- 为什么析构函数不能负责自动拆除一个已经不可达的环；
- 如何从业务关系中找出真正的 owner，并把反向边改成 `weak_ptr`；
- lambda 捕获 `shared_ptr` 如何形成隐蔽的 self-cycle；
- `enable_shared_from_this`、`shared_from_this()` 与 `weak_from_this()` 的正确边界；
- callback 跨 C ABI/Kotlin/Native 时需要怎样的生命周期协议。

所有失败场景都会在 Notebook 中受控拆环，避免给 kernel 留下真实泄漏。

In [ ]:
// 本步骤：引入引用计数、回调、字符串、断言和移动所需的标准库。
#include <cassert>
#include <functional>
#include <iostream>
#include <memory>
#include <string>
#include <utility>

### 1. 两条 strong edge 构成循环 ownership

初稿中的 A 和 B 都用 `shared_ptr` 保存对方。这不是普通的“双向关联”，而是在类型层面声明双方互相决定对方的生命周期。

In [ ]:
// 本步骤：前置声明 StrongB，使 StrongA 可以保存指向它的 shared_ptr。
class StrongB;

// 本步骤：定义 strong-own B 的 A，并记录生命周期。
class StrongA
{
public:
    StrongA()
    {
        // 构造时增加计数，用于确认循环存在时对象仍存活。
        ++live_count_;
        std::cout << "StrongA created, live = " << live_count_ << '\n';
    }

    ~StrongA() noexcept
    {
        // 只有 strong count 能降到零时，析构函数才会开始执行。
        --live_count_;
        std::cout << "StrongA destroyed, live = " << live_count_ << '\n';
    }

    void connect(std::shared_ptr<StrongB> b)
    {
        // 保存 strong owner，声明 A 参与决定 B 的生命周期。
        b_ = std::move(b);
    }

    std::shared_ptr<StrongB> b() const
    {
        // 返回 shared_ptr 副本，供受控实验取得 B ownership。
        return b_;
    }

    void disconnect() noexcept
    {
        // 主动删除 A -> B 的 strong edge。
        b_.reset();
    }

    static int live_count() noexcept
    {
        // 返回仍存活的 StrongA 数量。
        return live_count_;
    }

########33啊####    std::shared_ptr<StrongB> b_;
    inline static int live_count_ = 0;
};

// 本步骤：定义 strong-own A 的 B，完成双向 ownership 环。
class StrongB
{
public:
    StrongB()
    {
        // 构造时增加计数，记录 B 的生命周期。
        ++live_count_;
        std::cout << "StrongB created, live = " << live_count_ << '\n';
    }

    ~StrongB() noexcept
    {
        // B 的 strong count 归零后才会析构。
        --live_count_;
        std::cout << "StrongB destroyed, live = " << live_count_ << '\n';
    }

    void connect(std::shared_ptr<StrongA> a)
    {
        // 保存 strong owner，建立 B -> A 的反向 ownership。
        a_ = std::move(a);
    }

    void disconnect() noexcept
    {
        // 主动删除 B -> A 的 strong edge。
        a_.reset();
    }

    static int live_count() noexcept
    {
        // 返回仍存活的 StrongB 数量。
        return live_count_;
    }

private:
    std::shared_ptr<StrongA> a_;
    inline static int live_count_ = 0;
};

In [ ]:
// 本步骤：建立 A <-> B strong cycle，并计算外部 owner 消失后的引用计数。
{
    const int before_a = StrongA::live_count();
    const int before_b = StrongB::live_count();
    auto a = std::make_shared<StrongA>();
    auto b = std::make_shared<StrongB>();

    // weak observer 不参与循环，仅用于外部 owner 消失后验证对象仍存活。
    std::weak_ptr<StrongA> observe_a = a;
    std::weak_ptr<StrongB> observe_b = b;

    // 两条 strong edge 建立后，每个对象都有一个外部 owner 和一个内部 owner。
    a->connect(b);
    b->connect(a);
    assert(a.use_count() == 2);
    assert(b.use_count() == 2);

    // 释放外部 owner 后，每个对象仍被对方 strong-own，计数无法归零。
    a.reset();
    b.reset();
    assert(StrongA::live_count() == before_a + 1);
    assert(StrongB::live_count() == before_b + 1);
    assert(observe_a.use_count() == 1);
    assert(observe_b.use_count() == 1);
    std::cout << "external owners gone; cycle still alive\n";

    // 教学环境通过 weak observer 救回两个节点，再明确删除两条 strong edge。
    std::shared_ptr<StrongA> rescued_a = observe_a.lock();
    std::shared_ptr<StrongB> rescued_b = rescued_a->b();
    rescued_a->disconnect();
    rescued_b->disconnect();

    // 释放救援 owner 后，A 和 B 才终于能够析构。
    rescued_b.reset();
    rescued_a.reset();
    assert(StrongA::live_count() == before_a);
    assert(StrongB::live_count() == before_b);
    assert(observe_a.expired());
    assert(observe_b.expired());
}

为什么析构函数不能自动修复这个环？因为释放 `a` 和 `b` 外部变量后，两个 strong count 都仍为 1，析构函数根本不会开始执行。把 `disconnect()` 写进析构函数没有帮助：只有环已经被其他机制拆除，析构才有机会运行。

真实程序如果没有预留 observer 或显式 shutdown 路径，这组对象会保持不可达但存活，直到进程结束。内存诊断工具可能报告泄漏，但根因是 ownership graph 设计错误。

### 2. 先确定真正的 ownership 方向

对象可以互相访问，不代表必须互相拥有。假设 `OwnerA` 的确负责 `ObservedB` 的生命周期，而 B 只需回看 A，则关系应为：

```text
OwnerA -- shared ownership --> ObservedB
OwnerA <-- weak observation -- ObservedB
```

weak edge 仍保留关联能力，但不会阻止真正的 owner 被销毁。

In [ ]:
// 本步骤：前置声明 OwnerA，使 ObservedB 能保存 weak_ptr<OwnerA>。
class OwnerA;

// 本步骤：定义只观察 A 的 B，不参与决定 A 的生命周期。
class ObservedB
{
public:
    ObservedB()
    {
        // 记录 B 的构造，供修复后的清理断言使用。
        ++live_count_;
        std::cout << "ObservedB created, live = " << live_count_ << '\n';
    }

    ~ObservedB() noexcept
    {
        // A 释放 strong child 后，B 可以正常析构。
        --live_count_;
        std::cout << "ObservedB destroyed, live = " << live_count_ << '\n';
    }

    void observe(const std::shared_ptr<OwnerA> &a) noexcept
    {
        // 保存 weak edge，不增加 A 的 strong count。
        a_ = a;
    }

    std::shared_ptr<OwnerA> lock_a() const noexcept
    {
        // 使用前尝试建立临时 strong owner，过期时返回空。
        return a_.lock();
    }

    static int live_count() noexcept
    {
        // 返回当前存活的 ObservedB 数量。
        return live_count_;
    }

private:
    std::weak_ptr<OwnerA> a_;
    inline static int live_count_ = 0;
};

// 本步骤：定义真正 strong-own B 的 A。
class OwnerA
{
public:
    OwnerA()
    {
        // 记录 A 的构造，观察 ownership root 的生命周期。
        ++live_count_;
        std::cout << "OwnerA created, live = " << live_count_ << '\n';
    }

    ~OwnerA() noexcept
    {
        // 外部 root 消失时 A 能立即析构，并释放成员 b_。
        --live_count_;
        std::cout << "OwnerA destroyed, live = " << live_count_ << '\n';
    }

    void own(std::shared_ptr<ObservedB> b)
    {
        // 保存唯一的反向 strong edge A -> B。
        b_ = std::move(b);
    }

    static int live_count() noexcept
    {
        // 返回当前存活的 OwnerA 数量。
        return live_count_;
    }

private:
    std::shared_ptr<ObservedB> b_;
    inline static int live_count_ = 0;
};

In [ ]:
// 本步骤：用 A strong-own B、B weak-observe A 的结构验证自动清理。
{
    const int before_a = OwnerA::live_count();
    const int before_b = ObservedB::live_count();
    auto a = std::make_shared<OwnerA>();
    auto b = std::make_shared<ObservedB>();

    // A 保存 B 的 strong owner，B 只保存回看 A 的 weak observer。
    a->own(b);
    b->observe(a);
    assert(a.use_count() == 1);
    assert(b.use_count() == 2);
    assert(b->lock_a().get() == a.get());

    // 释放 B 的外部 owner后，B 仍由 A 保活。
    b.reset();
    assert(ObservedB::live_count() == before_b + 1);

    // 释放真正的 root owner A，A 析构后带动 B 析构。
    a.reset();
    assert(OwnerA::live_count() == before_a);
    assert(ObservedB::live_count() == before_b);
}

### 3. 回调捕获也可能形成 self-cycle

SDK 中更隐蔽的循环常出现在回调：对象拥有一个 `std::function`，lambda 又按值捕获该对象的 `shared_ptr`。

```text
Worker -- owns --> callback lambda -- shared ownership --> Worker
```

对象需要从成员函数取得自己的现有 control block 时，可继承 `std::enable_shared_from_this<T>`。绝不能写 `std::shared_ptr<Worker>(this)`；那会创建第二个 control block，最终导致 double delete。

In [ ]:
// 本步骤：定义能安装 strong-capture 或 weak-capture 回调的 Worker。
class Worker : public std::enable_shared_from_this<Worker>
{
public:
    explicit Worker(std::string name)
        : name_(std::move(name))
    {
        // 记录 Worker 生命周期，检查回调是否造成保活。
        ++live_count_;
        std::cout << "Worker " << name_ << " created, live = " << live_count_ << '\n';
    }

    ~Worker() noexcept
    {
        // self-cycle 存在时，这个析构函数不会自动执行。
        --live_count_;
        std::cout << "Worker " << name_ << " destroyed, live = " << live_count_ << '\n';
    }

    void install_strong_callback()
    {
        // shared_from_this 取得同一 control block 的 strong owner。
        std::shared_ptr<Worker> self = shared_from_this();

        // lambda 保存 self 副本，而 lambda 又由 Worker 成员拥有，形成 self-cycle。
        callback_ = [self]()
        {
            std::cout << "strong callback on " << self->name_ << '\n';
        };
    }

    void install_weak_callback()
    {
        // weak_from_this 取得同一 control block 的 observer，不增加 strong count。
        std::weak_ptr<Worker> self = weak_from_this();

        // 回调执行时才 lock；对象过期就跳过工作。
        callback_ = [self]()
        {
            if (std::shared_ptr<Worker> locked = self.lock())
            {
                std::cout << "weak callback on " << locked->name_ << '\n';
            }
            else
            {
                std::cout << "Worker expired; callback skipped\n";
            }
        };
    }

    void run_callback() const
    {
        // callback 存在时同步执行；回调自身决定能否取得 Worker。
        if (callback_)
        {
            callback_();
        }
    }

    void clear_callback() noexcept
    {
        // 销毁 lambda；strong capture 存在时，这一步会删除 self-cycle edge。
        callback_ = nullptr;
    }

    static int live_count() noexcept
    {
        // 返回当前存活的 Worker 数量。
        return live_count_;
    }

private:
    std::string name_;
    std::function<void()> callback_;
    inline static int live_count_ = 0;
};

In [ ]:
// 本步骤：安装 strong-capture 回调，受控复现并拆除 Worker self-cycle。
{
    const int before = Worker::live_count();
    auto worker = std::make_shared<Worker>("strong");
    std::weak_ptr<Worker> observer = worker;

    // 安装回调后，外部变量和 lambda capture 都是 strong owner。
    worker->install_strong_callback();
    assert(worker.use_count() == 2);
    worker->run_callback();

    // 外部 owner 消失后，callback -> Worker 的 strong edge 仍让对象存活。
    worker.reset();
    assert(Worker::live_count() == before + 1);
    assert(!observer.expired());

    // 教学环境重新取得 Worker，清空 callback 以删除 self-cycle。
    std::shared_ptr<Worker> rescued = observer.lock();
    rescued->clear_callback();
    rescued.reset();
    assert(Worker::live_count() == before);
    assert(observer.expired());
}

### 4. callback 默认捕获 weak observer

如果 callback 不应决定对象生命周期，就捕获 `weak_ptr`。执行回调时 `lock()`：成功则在本次调用期间安全使用对象，失败则取消工作。这种模式常用于异步任务、listener 和事件队列。

In [ ]:
// 本步骤：安装 weak-capture 回调，确认回调不会增加 Worker strong count。
{
    const int before = Worker::live_count();
    auto worker = std::make_shared<Worker>("weak");
    std::weak_ptr<Worker> observer = worker;

    // weak capture 不增加引用计数，唯一 strong owner 仍是外部 worker。
    worker->install_weak_callback();
    assert(worker.use_count() == 1);
    worker->run_callback();
    assert(worker.use_count() == 1);

    // 释放唯一 strong owner 后，callback 不会阻止 Worker 析构。
    worker.reset();
    assert(Worker::live_count() == before);
    assert(observer.expired());
}

`shared_from_this()` 只能在对象已经由 `shared_ptr` 管理后调用。在构造函数中调用，或对栈对象调用，会因为尚未关联有效 control block 而抛出 `std::bad_weak_ptr`。`weak_from_this()` 在这种情况下返回空 observer，但通常仍应在工厂完成 shared ownership 后再安装回调。

### 5. Ownership graph 设计检查

遇到双向关系时，不要机械地把一端换成 `weak_ptr`，先回答谁负责谁：

| 关系 | 推荐 edge | 原因 |
| --- | --- | --- |
| parent → child | strong / `unique_ptr` 或 `shared_ptr` | parent 决定 child 生命周期 |
| child → parent | `weak_ptr` 或受限 raw pointer/reference | 只回看，不应保活 parent |
| registry/cache → object | 通常 `weak_ptr` | 登记不应阻止对象销毁 |
| async callback → receiver | 通常 weak capture | 任务不应无意延长 receiver 生命周期 |
| active job → immutable request | 可以 strong | job 明确需要 request 活到任务结束 |

对象图描述“谁认识谁”，ownership graph 描述“谁保证谁活着”。只有后者决定智能指针类型。

### 6. Native SDK 与跨语言 callback

`shared_ptr`、`weak_ptr`、lambda 和 `std::function` 都是 C++ 类型，不能直接穿过 C ABI。C callback 常被拆成函数指针与 context：

```c
typedef void (*sdk_callback)(void *context, sdk_status status);

sdk_status sdk_set_callback(
    sdk_handle *handle,
    sdk_callback callback,
    void *context
);
```

但 `void*` 不携带生命周期信息。API 必须明确：谁拥有 context、callback 最晚何时可能触发、取消函数是否等待在途回调结束、销毁 handle 后是否保证不再回调。C++ 内部可以用 weak capture 防止访问已销毁对象；Kotlin/Native wrapper 则需要按 cinterop 规则管理稳定引用，并在取消或销毁后释放它。

跨 ABI 的安全来自完整的注册、取消和销毁协议，不能只靠把某个 raw pointer 传给回调。

## 实验结论

循环引用的根因不是 `shared_ptr` 实现错误，而是 ownership graph 中存在无法从外部解除的 strong cycle。引用计数不是 tracing garbage collector，不会分析一组对象是否已经不可达。

需要直接形成以下判断：

```text
strong edge
→ 我保证目标对象继续存活

weak edge
→ 我只需要找到目标对象
→ 不决定它何时销毁
→ 使用前必须 lock()
```

验收要点：

1. 外部 owner 消失不等于 strong count 必然归零；
2. 析构函数无法拆除一个阻止析构本身发生的环；
3. 双向访问不等于双向 ownership；
4. callback 按值捕获 `shared_ptr` 也会成为 strong edge；
5. `weak_ptr` 和 weak capture 用于不应保活目标的反向关系；
6. C ABI callback 必须另外定义 context 的注册、取消与销毁时序。